# Equipo - Accidentes de tránsito en México
## Regresión Lineal y Regresión Logística con Python

**Objetivo del notebook:**  
Construir dos modelos analíticos con base en el dataset de accidentes por entidad federativa:

1. **Regresión lineal** para estimar la cantidad de accidentes mortales.
2. **Regresión logística** para clasificar si una entidad presenta alto riesgo mortal o no.

> Nota: Este análisis tiene fines académicos y exploratorios.  
> Debido al tamaño del dataset, los resultados deben interpretarse como una base metodológica y no como un sistema definitivo de predicción.

In [2]:
# ==========================================================
# 1. IMPORTACIÓN DE LIBRERÍAS Y CARGA DE DATOS
# ==========================================================
# En esta sección preparamos el entorno de trabajo.
# Cargamos librerías para:
# - manipular datos
# - visualizar información
# - entrenar modelos
# - evaluar resultados

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, roc_auc_score
)

sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

# ----------------------------------------------------------
# CARGA DEL ARCHIVO
# ----------------------------------------------------------
# Instrucción para el equipo:
# 1. Subir el archivo a Google Colab
# 2. Copiar la ruta
# 3. Pegarla aquí
#
# Este bloque detecta si el archivo es CSV o Excel.

file_path = "/content/accidentes_equipo.xlsx"   # <-- CAMBIAR RUTA

if file_path.endswith(".csv"):
    df = pd.read_csv(file_path)
elif file_path.endswith(".xlsx") or file_path.endswith(".xls"):
    df = pd.read_excel(file_path)
else:
    raise ValueError("Formato no soportado. Usa CSV o Excel.")

print("Dataset cargado correctamente.")
display(df.head())

FileNotFoundError: [Errno 2] No such file or directory: '/content/accidentes_equipo.xlsx'

In [ ]:
# ==========================================================
# 2. LIMPIEZA INICIAL
# ==========================================================
# Aquí normalizamos nombres de columnas y corregimos una posible
# inconsistencia común: que la segunda columna se llame "accidentes"
# pero realmente contenga el tipo de fila ("accidentes" o "accidentes mortales").

# Normalizar nombres de columnas
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

# Si la segunda columna se llama "accidentes" pero contiene texto como
# "accidentes" / "accidentes mortales", entonces la renombramos a "tipo".
if df.columns[1] == "accidentes":
    if df.iloc[:, 1].dtype == "object":
        df = df.rename(columns={df.columns[1]: "tipo"})

print("Columnas del dataset:")
print(df.columns.tolist())

# Columnas esperadas de conteo
dias = ["lunes", "martes", "miercoles", "jueves", "viernes", "sabado", "domingo"]
luces = ["luz_dia", "luz_crepusculo", "luz_noche", "luz_alumbrado_publico"]

# Convertir columnas numéricas
for col in dias + luces:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Estandarizar la columna tipo
df["tipo"] = df["tipo"].astype(str).str.strip().str.lower()

print("\nValores faltantes por columna:")
print(df.isnull().sum())

display(df.head())

In [ ]:
# ==========================================================
# 3. ANÁLISIS EXPLORATORIO INICIAL
# ==========================================================
# En esta etapa revisamos:
# - cuántos registros hay por tipo
# - si los totales por día y por condición de luz coinciden
# - cómo se distribuyen los accidentes y los accidentes mortales

print("Dimensiones del dataset:", df.shape)
print("\nTipos encontrados:")
print(df["tipo"].value_counts())

# Totales por fila
df["total_dias"] = df[dias].sum(axis=1)
df["total_luz"] = df[luces].sum(axis=1)

# Verificamos si lo contado por días coincide con lo contado por luz
df["coinciden_totales"] = df["total_dias"] == df["total_luz"]

print("\nValidación de consistencia (total días vs total luz):")
print(df["coinciden_totales"].value_counts())

display(df[["entidad_federativa", "tipo", "total_dias", "total_luz", "coinciden_totales"]].head(10))

# Visualización rápida
plt.figure(figsize=(8,4))
sns.boxplot(data=df, x="tipo", y="total_dias", palette="Set2")
plt.title("Distribución de totales por tipo")
plt.xlabel("Tipo de registro")
plt.ylabel("Total registrado")
plt.show()

In [ ]:
# ==========================================================
# 4. CONSTRUIR TABLA ANALÍTICA POR ENTIDAD
# ==========================================================
# El dataset viene en formato "dos filas por entidad":
# - una fila con accidentes
# - una fila con accidentes mortales
#
# Para modelar mejor, convertimos eso en una sola fila por entidad,
# donde:
# - las variables predictoras salen de la fila "accidentes"
# - la variable objetivo sale de la fila "accidentes mortales"

df_acc = df[df["tipo"] == "accidentes"].copy()
df_mort = df[df["tipo"].str.contains("mortales")].copy()

# Nos quedamos con la fila de accidentes y agregamos como target
# el total de accidentes mortales por entidad
modelo_df = df_acc[["entidad_federativa"] + dias + luces + ["total_dias"]].copy()
modelo_df = modelo_df.rename(columns={"total_dias": "total_accidentes"})

modelo_df = modelo_df.merge(
    df_mort[["entidad_federativa", "total_dias"]].rename(columns={"total_dias": "total_mortales"}),
    on="entidad_federativa",
    how="inner"
)

# ----------------------------------------------------------
# INGENIERÍA DE VARIABLES
# ----------------------------------------------------------
# Creamos variables más interpretables para negocio y modelado:
# - porcentaje de accidentes en fin de semana
# - porcentaje de accidentes nocturnos
# - porcentaje en crepúsculo
# - porcentaje con luz de día

modelo_df["pct_fin_semana"] = (
    modelo_df["viernes"] + modelo_df["sabado"] + modelo_df["domingo"]
) / modelo_df["total_accidentes"]

modelo_df["pct_noche"] = modelo_df["luz_noche"] / modelo_df["total_accidentes"]
modelo_df["pct_dia"] = modelo_df["luz_dia"] / modelo_df["total_accidentes"]
modelo_df["pct_crepusculo"] = modelo_df["luz_crepusculo"] / modelo_df["total_accidentes"]

print("Tabla analítica lista para modelado.")
display(modelo_df.head())

print("\nResumen estadístico:")
display(modelo_df.describe())

In [ ]:
# ==========================================================
# 5. VISUALIZACIÓN EXPLORATORIA CLAVE
# ==========================================================
# Este bloque ayuda a interpretar relaciones iniciales entre:
# - volumen total de accidentes
# - proporción nocturna
# - accidentes mortales

plt.figure(figsize=(7,5))
sns.scatterplot(data=modelo_df, x="total_accidentes", y="total_mortales", hue="pct_noche", palette="Reds", s=100)
plt.title("Accidentes totales vs accidentes mortales")
plt.xlabel("Total de accidentes")
plt.ylabel("Total de accidentes mortales")
plt.show()

plt.figure(figsize=(8,5))
corr_cols = ["total_accidentes", "pct_fin_semana", "pct_noche", "pct_dia", "pct_crepusculo", "total_mortales"]
sns.heatmap(modelo_df[corr_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlación entre variables clave")
plt.show()

# Regresión lineal

La regresión lineal busca estimar una variable numérica continua.

## Fórmula general
\[
\hat{Y} = \beta_0 + \beta_1X_1 + \beta_2X_2 + \cdots + \beta_nX_n
\]

## En este proyecto:
- \(\hat{Y}\) = accidentes mortales estimados
- \(X_1\) = total de accidentes
- \(X_2\) = proporción de accidentes en fin de semana
- \(X_3\) = proporción de accidentes nocturnos
- \(X_4\) = proporción de accidentes en crepúsculo

## Objetivo del modelo
Estimar cuántos accidentes mortales podría presentar una entidad en función de su volumen total de siniestros y de sus patrones temporales y de iluminación.

## Valor para negocio
Este modelo puede apoyar a aseguradoras o autoridades para:
- detectar entidades con mayor severidad esperada
- priorizar vigilancia y prevención
- ajustar pricing y evaluación de riesgo

In [ ]:
# ==========================================================
# 6. REGRESIÓN LINEAL
# ==========================================================
# Variable objetivo:
# - total_mortales
#
# Variables predictoras:
# - total_accidentes
# - pct_fin_semana
# - pct_noche
# - pct_crepusculo
#
# Separamos en entrenamiento y prueba para validar si el modelo
# puede generalizar a datos no vistos.

features_lin = ["total_accidentes", "pct_fin_semana", "pct_noche", "pct_crepusculo"]
X_lin = modelo_df[features_lin]
y_lin = modelo_df["total_mortales"]

X_train_lin, X_test_lin, y_train_lin, y_test_lin = train_test_split(
    X_lin, y_lin, test_size=0.25, random_state=42
)

modelo_lineal = LinearRegression()
modelo_lineal.fit(X_train_lin, y_train_lin)

# Coeficientes
coef_lineal = pd.DataFrame({
    "Variable": features_lin,
    "Coeficiente": modelo_lineal.coef_
})

print("Intercepto (β0):", round(modelo_lineal.intercept_, 4))
display(coef_lineal)

# Predicciones
y_pred_lin = modelo_lineal.predict(X_test_lin)

# Métricas
mae = mean_absolute_error(y_test_lin, y_pred_lin)
mse = mean_squared_error(y_test_lin, y_pred_lin)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_lin, y_pred_lin)

print("\nMétricas de regresión lineal")
print(f"MAE  : {mae:.4f}")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")

# Comparación real vs predicho
resultados_lin = pd.DataFrame({
    "Entidad": modelo_df.loc[X_test_lin.index, "entidad_federativa"].values,
    "Real": y_test_lin.values,
    "Predicho": y_pred_lin
})

display(resultados_lin)

In [ ]:
# ==========================================================
# 7. VISUALIZACIÓN DEL MODELO LINEAL
# ==========================================================
# Si el modelo ajusta razonablemente bien, los puntos deberían
# acercarse a la línea diagonal (predicción perfecta).

plt.figure(figsize=(7,5))
plt.scatter(y_test_lin, y_pred_lin, color="teal", s=90)
plt.plot([y_test_lin.min(), y_test_lin.max()],
         [y_test_lin.min(), y_test_lin.max()],
         linestyle="--", color="red")
plt.xlabel("Accidentes mortales reales")
plt.ylabel("Accidentes mortales predichos")
plt.title("Regresión lineal: reales vs predichos")
plt.show()

# Regresión logística

La regresión logística se usa cuando la variable objetivo es binaria.

## Idea probabilística
\[
P(Y=1|X)=\frac{1}{1+e^{-z}}
\]

donde:

\[
z = \beta_0 + \beta_1X_1 + \beta_2X_2 + \cdots + \beta_nX_n
\]

## Enfoque binario (Bernoulli / binomial)
Para aplicar regresión logística, necesitamos una variable con dos clases:

- 1 = entidad con **alto riesgo mortal**
- 0 = entidad con **riesgo mortal no alto**

En este notebook, construiremos esa variable usando la mediana de `total_mortales` como punto de corte.

## ¿Por qué esto tiene sentido?
Porque el problema del negocio no siempre es “¿cuántos muertos habrá exactamente?”, sino también:
- ¿esta entidad debe considerarse de alto riesgo?
- ¿requiere atención prioritaria?
- ¿debería ajustarse su pricing o estrategia preventiva?

In [ ]:
# ==========================================================
# 8. REGRESIÓN LOGÍSTICA
# ==========================================================
# Creamos una variable binaria:
# - 1 si total_mortales > mediana
# - 0 si total_mortales <= mediana
#
# Esto permite convertir el problema en clasificación.

umbral = modelo_df["total_mortales"].median()
modelo_df["alto_riesgo_mortal"] = (modelo_df["total_mortales"] > umbral).astype(int)

print("Umbral usado (mediana de total_mortales):", umbral)
print("\nDistribución de clases:")
print(modelo_df["alto_riesgo_mortal"].value_counts())

features_log = ["total_accidentes", "pct_fin_semana", "pct_noche", "pct_crepusculo"]
X_log = modelo_df[features_log]
y_log = modelo_df["alto_riesgo_mortal"]

X_train_log, X_test_log, y_train_log, y_test_log = train_test_split(
    X_log, y_log, test_size=0.25, random_state=42, stratify=y_log
)

# Escalado
scaler = StandardScaler()
X_train_log_scaled = scaler.fit_transform(X_train_log)
X_test_log_scaled = scaler.transform(X_test_log)

# Modelo
modelo_log = LogisticRegression(max_iter=1000)
modelo_log.fit(X_train_log_scaled, y_train_log)

# Predicción de clase y probabilidad
y_pred_log = modelo_log.predict(X_test_log_scaled)
y_prob_log = modelo_log.predict_proba(X_test_log_scaled)[:, 1]

# Coeficientes
coef_log = pd.DataFrame({
    "Variable": features_log,
    "Coeficiente": modelo_log.coef_[0]
})

print("\nIntercepto logístico (β0):", round(modelo_log.intercept_[0], 4))
display(coef_log)

# Métricas
acc = accuracy_score(y_test_log, y_pred_log)
prec = precision_score(y_test_log, y_pred_log, zero_division=0)
rec = recall_score(y_test_log, y_pred_log, zero_division=0)
f1 = f1_score(y_test_log, y_pred_log, zero_division=0)
auc = roc_auc_score(y_test_log, y_prob_log)

print("\nMétricas de regresión logística")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"AUC      : {auc:.4f}")

print("\nClassification report:")
print(classification_report(y_test_log, y_pred_log, zero_division=0))

In [ ]:
# ==========================================================
# 9. EVALUACIÓN VISUAL DE LA CLASIFICACIÓN
# ==========================================================
# La matriz de confusión permite observar:
# - TP: verdaderos positivos
# - TN: verdaderos negativos
# - FP: falsos positivos
# - FN: falsos negativos

cm = confusion_matrix(y_test_log, y_pred_log)

plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Matriz de confusión - Regresión logística")
plt.xlabel("Predicción")
plt.ylabel("Valor real")
plt.show()

# Curva ROC
fpr, tpr, thresholds = roc_curve(y_test_log, y_prob_log)

plt.figure(figsize=(7,5))
plt.plot(fpr, tpr, label=f"AUC = {auc:.4f}", color="darkblue")
plt.plot([0,1], [0,1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Curva ROC - Riesgo mortal")
plt.legend()
plt.show()

# Interpretación final

## Regresión lineal
Este modelo estima cuántos accidentes mortales puede presentar una entidad a partir de:
- el volumen total de accidentes
- la proporción de eventos en fin de semana
- la proporción de eventos nocturnos
- la proporción de eventos en crepúsculo

## Regresión logística
Este modelo clasifica si una entidad debe considerarse de **alto riesgo mortal** o no.

## Relación con el proyecto
Estos dos enfoques apoyan el objetivo del equipo porque permiten:
- identificar entidades con mayor severidad vial
- justificar estrategias de prevención
- proponer segmentación de riesgo para aseguradoras
- traducir datos históricos en decisiones de negocio

## Importante
Dado que el número de observaciones es pequeño, este ejercicio debe presentarse como:
- una base metodológica válida
- un modelo exploratorio
- una propuesta inicial que podría fortalecerse con más granularidad (por mes, municipio o año)